In [ ]:
import sys

from corner import corner

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.flow_matching.distributions import DiscreteUniform
from src.flow_matching.helpers import choose_device
from src.flow_matching.loader import (
    load_model_info,
    load_trained_model,
    prob_path_from_config,
)
from src.flow_matching.neural_posterior import NeuralPosterior
from src.flow_matching.simulator import Model as BatchedModel
from src.helpers import gen_parameter_labels

device = choose_device()

# Loading in the FM model

In [ ]:
# fill in desired job_id or directory name 
job_id = "15101905"

# loading the model (found via job_id, or path)
vector_field, classifier, config = load_trained_model(job_id, device, ema=False)

transdimensional = not config['training']['fixed_N']
path = prob_path_from_config(config)
model_info = load_model_info(config, path=path, device=device)

In [ ]:
inf_params  = model_info['inf_params']
N           = model_info['N']
vector_dim  = model_info['vector_dim']
burstparams = model_info['burstparams']
mean        = model_info['mean']
std         = model_info['std']
YBKG        = config['path']['p_data']['init_params']['model_params']['ybkg']

# generate FM posterior conditioned on simulated counts

In [ ]:
# sample lightcurve to condition on
params, simulated_counts, components = path.p_data.sample(1)
burstparams = path.p_data.prior.samples_as_dict(params)
N_true = components[0][0].item()

print("This example has N = ", components[0][0].item(), " components")
print(
    f"""
    The burst parameters are:
    t0 =\t {np.array_str(burstparams['t0'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    log(amp) =\t {np.array_str(burstparams['amp'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    log(rise) =\t {np.array_str(burstparams['rise'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    skew =\t {np.array_str(burstparams['skew'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    """
)
simulated_counts = simulated_counts[0]
plt.plot(simulated_counts.cpu().numpy())
plt.show()

In [ ]:
learned_posterior = NeuralPosterior(
    vector_field,
    classifier,
    model_info,
    path,
    device=device
)

In [ ]:
num_samples = 10000
samples_per_batch = 5000

Ns, theta_samples = learned_posterior.sample(num_samples, simulated_counts, samples_per_batch, time_steps=200)
p_N = learned_posterior.p_N
Ns = Ns.to('cpu')
theta_samples

# Classifier p(N)

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

print("True number of components is: ", components[0][0].item())
print("Most likely estimate from classifier: ", range(1, len(p_N.cpu().detach().numpy())+1)[torch.where(p_N == torch.max(p_N))[0]])

# corner plot (for a given N)

In [ ]:
range_ = 0.95
N_inf = 7 # choose for which N to make corner plot (for FM)
N_inf = N if not transdimensional else N_inf 

FM_samples = theta_samples[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf

# make vector correct size 
# (f.e. if N=2 select t0_1, t0_2, rise_1, rise_2 from vector structured like [t0_1 ... t0_Nmax,  rise_1 ... rise_Nmax])
bs, dim = FM_samples.shape
new_dim = N_inf * len(inf_params)

FM_samples = FM_samples.view(-1, len(inf_params), N)[:, :, :N_inf].reshape(bs, new_dim)  

var_names = gen_parameter_labels(inf_params, N)
var_names = np.array(var_names).reshape(len(inf_params), N)[:, :N_inf].flatten()

t0_labels = [f'$t_0^{i+1}$' for i in range(N_inf)]
log_r_labels = [f'$\log(r_{i+1})$' for i in range(N_inf)]
s_labels = [f'$s_{i+1}$' for i in range(N_inf)]
log_A_labels = [f'$\log(A_{i+1})$' for i in range(N_inf)]

true_values = None
if N_inf == N_true:
    true_values = np.array([burstparams[key][0][:N_inf] for key in inf_params]).flatten()
    # true_values = true_values.reshape(len(inf_params), N_inf)

fig = corner(
    FM_samples.cpu().numpy(), 
    labels=t0_labels + log_r_labels + s_labels + log_A_labels, 
    truths=true_values, 
    range=[range_ for _ in range(N_inf * len(inf_params))], 
    color='black', 
    label='FM', 
    plot_density=True, 
    plot_datapoints=True, 
    fill_contours=False, 
    plot_contours=True, 
    hist_kwargs={'density':True}
    )

plt.show()

## posterior samples

In [ ]:
# posterior samples that include all sampled component numbers
modelparams = {'time':torch.linspace(0, 1, simulated_counts.shape[0]), 'burstparams': burstparams, 'ybkg':0,'ncomp':components}

true_flux = BatchedModel(**modelparams, device=device).get_flux().flatten()

plt.figure(figsize=(20,5))
plt.subplot(121)
plt.plot(simulated_counts.flatten().cpu(),'k-',alpha=0.5,  label='noisy flux')

posterior_param_samples = path.p_data.prior.samples_as_dict(theta_samples)
# Ns_10 = torch.ones_like(Ns) * 2
posterior_curve_samples = BatchedModel(
    device=device, 
    time=torch.linspace(0, 1, simulated_counts.shape[0]), 
    burstparams=posterior_param_samples, 
    ybkg=YBKG, 
    ncomp=Ns
        ).get_flux()

for i in range(100):
    random_idx = torch.randint(0, num_samples, size=(1,)).item()
    plt.plot(posterior_curve_samples[random_idx].cpu(), alpha=0.75)
plt.title('100 posterior samples')
plt.subplot(122)
plt.plot(simulated_counts.flatten().cpu(),alpha=0.5, label='poisson flux')
plt.plot(true_flux.cpu(), 'k--',label='ground truth')
plt.legend()

plt.show()
print(simulated_counts.shape)

---
# Averaging $p_\xi(N|y)$ over a collection of lightcurves

In [ ]:
examples = 2000

path.p_data.device = device
path = path.to(device)

for key in path.p_data.prior.priors:
        path.p_data.prior.priors[key].device=device
path.p_data.N_prior.device=device

plt.figure(figsize=(15, 3))
for i, n in enumerate([1, 3, 5, 8]):
    plt.subplot(1,4,i+1)
    path.p_data.N_prior = DiscreteUniform(n,n, device=device)
    
    path = path.to(device)
    params, counts, components = path.p_data.sample(examples)
    burstparams = path.p_data.prior.samples_as_dict(params)

    model = BatchedModel(
                time=torch.linspace(0, 1, 1000), 
                ncomp=components, 
                ybkg=YBKG, 
                burstparams=burstparams,
                device=device
                )
    true_flux = model.get_flux()
    condition = torch.tensor(counts / 150, device=device, dtype=torch.float)

    logits = classifier(condition)
    p_N = torch.softmax(logits, dim=1)
    colors = ['black' for i in range(10)]
    colors[n-1]='tab:green'

    plt.bar(range(1,p_N.shape[1]+1), torch.mean(p_N, axis=0).flatten().cpu().detach(), color=colors, alpha=0.6)

    plt.title(f'$N_{{true}}$ = {n}')#, x=0.85, y=0.9)

    plt.ylabel('⟨p(N | y)⟩', fontsize=14) if i == 0 else None
    plt.xlabel('N', fontsize=14) 

    plt.yticks(ticks=[0.2, 0.4, 0.6, 0.8, 1.0], labels=([] if i > 0 else None), fontsize=12)
    plt.xticks(fontsize=12)
    plt.ylim(0,1)
    # plt.title(f'average p(N|y) for {examples} sample curves with N={n} components')

    
# plt.tight_layout()
plt.show()

# Plotting $p_\xi(N|y)$ for some individual flux profiles

In [ ]:
# lightcurve vs classifier result
for i in range(1800, 1811):
        plt.figure(figsize=(15, 5))
        plt.subplot(121)
        plt.bar(range(1, p_N.shape[1]+1), height=p_N[i].flatten().cpu().detach().numpy())
        plt.xlabel("$N_{pred}$")
        plt.title('p(N|y)')

        N_true = components[i].item()
        true_parameters = path.p_data.prior.samples_as_dict(params[i][:N_true].view(1,-1))

        plt.subplot(122)
        plt.title(f'N_true = {N_true}')
        plt.plot(torch.linspace(0, 1, 1000), counts[i].cpu(), 'k-', label='simulated flux')
        plt.vlines(true_parameters['t0'].flatten(), ymin= torch.zeros_like(true_parameters['t0'].flatten()), ymax=torch.max(counts[i]).repeat(N_true).cpu(), linestyle='dotted', label='t_0')
        plt.plot(torch.linspace(0, 1, 1000), true_flux[i].cpu(), 'r-', alpha=0.7, label='ground truth')
        plt.legend(bbox_to_anchor=(1.3, 1.02))